# RAG Assignment: Baseline vs Re-Ranking (CATAN Rulebook)



In [17]:
# Cell 1 — install dependencies (Google Colab)
import sys
in_colab = 'google.colab' in sys.modules
if in_colab:
    get_ipython().system('pip -q install -U sentence-transformers pypdf google-genai transformers accelerate')
    print('Dependencies installed/updated.')
else:
    print('Not Colab — ensure sentence-transformers, pypdf, google-genai, transformers, and accelerate are installed.')


Dependencies installed/updated.


In [18]:
# Cell 2 — imports + central config (single place to change chunking / models / k)
import os, re, time, urllib.request
from pathlib import Path
from dataclasses import dataclass

import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder

PDF_URL = 'https://cdn.1j1ju.com/medias/7a/18/fd-catan-rulebook.pdf'  # Catan Game Rules & Almanac (2015 ed., 16 pp)
PDF_PATH = Path('data/catan_rulebook.pdf')

BI_ENCODER_NAME = 'sentence-transformers/all-MiniLM-L6-v2'   # dense bi-encoder, 384-d
CROSS_ENCODER_NAME = 'cross-encoder/ms-marco-MiniLM-L-6-v2'   # joint query+chunk scorer (as per brief)

CHUNK_SIZE = 500       # characters per chunk — small enough that a general rule and its
CHUNK_OVERLAP = 100    # 'except...' clause land in DIFFERENT chunks (needed for the demo)
TOP_K_CANDIDATES = 10  # brief: retrieve Top-10
TOP_K_FINAL = 3        # brief: feed Top-3 to the LLM

print('Config:', BI_ENCODER_NAME, '|', CROSS_ENCODER_NAME, f'| chunk={CHUNK_SIZE}/{CHUNK_OVERLAP}')

Config: sentence-transformers/all-MiniLM-L6-v2 | cross-encoder/ms-marco-MiniLM-L-6-v2 | chunk=500/100


In [19]:
# Cell 3 — download PDF (if missing) + extract text page by page (keeps page numbers for citations)
PDF_PATH.parent.mkdir(parents=True, exist_ok=True)
if not PDF_PATH.exists():
    print('Downloading CATAN rulebook...')
    urllib.request.urlretrieve(PDF_URL, PDF_PATH)
print(f'PDF: {PDF_PATH} ({PDF_PATH.stat().st_size/1e6:.1f} MB)')

reader = PdfReader(str(PDF_PATH))
pages = [(i + 1, (p.extract_text() or '')) for i, p in enumerate(reader.pages)]
pages = [(n, re.sub(r'\s+', ' ', t).strip()) for n, t in pages if t.strip()]
print(f'Pages with text: {len(pages)} | total chars: {sum(len(t) for _, t in pages)}')
print('Sample (p.2):', pages[1][1][:200], '...')

PDF: data/catan_rulebook.pdf (6.9 MB)
Pages with text: 16 | total chars: 53583
Sample (p.2): 2 Dear Settlers, To make it as easy as possible for you to start playing Catan®, we use an award-winning rules system, which consists of 3 parts. First, if you do not know how to play Catan, please re ...


In [20]:
# Cell 4 — chunking: fixed-size char windows with overlap, tagged with page + chunk id.
# WHY char windows and not semantic splitting? The brief requires the SAME corpus for both
# pipelines and wants general-rule vs exception chunks to separate — fixed windows guarantee that
# deterministically, while semantic splitting could merge them and hide the failure mode.
@dataclass
class Chunk:
    cid: str        # e.g. 'C042'
    page: int
    text: str

def chunk_pages(pages, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks, n = [], 0
    for pageno, text in pages:
        start = 0
        while start < len(text):
            piece = text[start:start + size].strip()
            if len(piece) > 80:  # drop tiny tail fragments
                chunks.append(Chunk(cid=f'C{n:03d}', page=pageno, text=piece))
                n += 1
            start += size - overlap
    return chunks

chunks = chunk_pages(pages)
print(f'Chunks: {len(chunks)} | avg len: {np.mean([len(c.text) for c in chunks]):.0f} chars')
for c in chunks[8:11]:
    print(f'--- {c.cid} (p.{c.page}) --- {c.text[:150]}...')

Chunks: 138 | avg len: 476 chars
--- C008 (p.3) --- 4 cities, and 15 roads (no more and no less!). Place your 2 roads and your 2 settlements on the game board. Place your remaining settlements, roads, a...
--- C009 (p.3) --- rmy” beside the game board along with the 2 dice. Sort the resource cards into 5 stacks and put them face up next to the game board. Shuffle the devel...
--- C010 (p.3) --- • 10 ••• 5 •••• 12 • 5 •••• 4 ••• 11 •• 10 ••• 11 •• 6 ••••• 4 ••• 9 •••• 6 ••••• 3 •• 3 •• 8 ••••• Illustration A Starting Map for Beginners To make ...


In [21]:
# Cell 5 — PIPELINE 1 index: Bi-Encoder embeddings (independent query/chunk vectors, cosine sim).
# Embeddings are L2-normalised once, so cosine similarity == dot product (fast, exact).
t0 = time.perf_counter()
bi = SentenceTransformer(BI_ENCODER_NAME)
corpus_emb = bi.encode([c.text for c in chunks], normalize_embeddings=True, show_progress_bar=True)
corpus_emb = np.asarray(corpus_emb, dtype=np.float32)
print(f'Indexed {len(chunks)} chunks in {time.perf_counter()-t0:.1f}s | dim={corpus_emb.shape[1]}')

def baseline_top10(query):
    """Return [(chunk_idx, cosine_score)] for Top-10 — shared candidate set for BOTH pipelines."""
    q = bi.encode([query], normalize_embeddings=True)
    scores = (np.asarray(q, dtype=np.float32) @ corpus_emb.T)[0]
    top = np.argsort(-scores)[:TOP_K_CANDIDATES]
    return [(int(i), float(scores[i])) for i in top]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Indexed 138 chunks in 2.4s | dim=384


In [22]:
# Cell 6 — PIPELINE 2 scorer: Cross-Encoder (joint self-attention over query+chunk tokens).
# Scores all 10 (query, chunk) pairs and re-sorts descending. NOT run over the whole corpus
# (see Scaling answer in the final cell + report: O(N) transformer forwards is infeasible).
cross = CrossEncoder(CROSS_ENCODER_NAME)
print('Cross-encoder loaded:', CROSS_ENCODER_NAME)

def rerank_top10(query, top10):
    """Return [(chunk_idx, cross_score)] re-sorted desc by Cross-Encoder relevance."""
    pairs = [(query, chunks[i].text) for i, _ in top10]
    scores = cross.predict(pairs)
    order = np.argsort(-np.asarray(scores))
    return [(top10[i][0], float(scores[i])) for i in order]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Cross-encoder loaded: cross-encoder/ms-marco-MiniLM-L-6-v2


In [23]:

# Cell 7 — Gemini LLM layer
# Both RAG pipelines use the same Gemini model and identical prompt structure.

import os
import time
from google import genai

# --------------------------------------------------
# 1. Load the API key securely
# --------------------------------------------------
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "").strip()

if not GEMINI_API_KEY:
    try:
        from google.colab import userdata
        GEMINI_API_KEY = (userdata.get("GEMINI_API_KEY") or "").strip()
    except Exception:
        pass

if not GEMINI_API_KEY:
    raise RuntimeError(
        "GEMINI_API_KEY is missing. Add it to Colab Secrets, "
        "enable notebook access, and rerun the cell."
    )

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

# --------------------------------------------------
# 2. One reusable Gemini client
# --------------------------------------------------
_gemini_client = genai.Client(api_key=GEMINI_API_KEY)

# --------------------------------------------------
# 3. Identical prompt structure for both pipelines
# --------------------------------------------------
PROMPT = """You are an assistant answering questions about the Catan board-game rules.

Use only the information explicitly supported by the context below.
Do not use outside knowledge or invent rules.
Pay careful attention to exceptions, restrictions, negation, and timing.
If the context does not contain enough information to answer, say:
"Not stated in the provided rules."
Answer in 2-4 concise sentences and cite the supporting chunk IDs,
using citations such as [C012].

Context:
{ctx}

Question: {q}

Answer:"""


def fmt_ctx(idxs):
    """Format retrieved chunk indices as cited context."""
    return "\n\n".join(
        f"[{chunks[i].cid} p.{chunks[i].page}] {chunks[i].text}"
        for i in idxs
    )


# --------------------------------------------------
# 4. Discover and test an available Gemini Flash model
# --------------------------------------------------
GEMINI_PREFERRED_MODELS = [
    "gemini-flash-latest",
    "gemini-2.5-flash",
    "gemini-2.5-flash-lite",
]

def _normalise_model_name(name):
    return str(name or "").removeprefix("models/")


def _discover_gemini_models():
    """Return available Gemini Flash model names."""
    models = list(_gemini_client.models.list())
    names = [
        _normalise_model_name(getattr(model, "name", ""))
        for model in models
    ]

    names = [
        name for name in names
        if name
        and "gemini" in name.lower()
        and "flash" in name.lower()
        and not any(
            term in name.lower()
            for term in ("image", "tts", "embedding", "robotics")
        )
    ]

    # Prefer the model that passed the earlier Cell 1b test, if present.
    previously_tested = os.environ.get("GEMINI_WORKING_MODEL", "").strip()
    ordered = []

    if previously_tested and previously_tested in names:
        ordered.append(previously_tested)

    for name in GEMINI_PREFERRED_MODELS + names:
        name = _normalise_model_name(name)
        if name in names and name not in ordered:
            ordered.append(name)

    return ordered


def _test_and_select_gemini():
    """Test models until one returns a real response."""
    candidates = _discover_gemini_models()

    if not candidates:
        raise RuntimeError(
            "No Gemini Flash models were found for this API key. "
            "Check API access and available models."
        )

    errors = []

    for model_name in candidates:
        try:
            response = _gemini_client.models.generate_content(
                model=model_name,
                contents=(
                    "Reply with exactly: GEMINI_TEST_OK"
                ),
                config={"temperature": 0.0, "max_output_tokens": 32},
            )

            answer = (response.text or "").strip()

            if answer:
                print("Gemini model test passed.")
                print("Selected model:", model_name)
                print("Test response:", answer[:100])
                return model_name

            errors.append(f"{model_name}: empty response")

        except Exception as exc:
            message = str(exc).replace("\n", " ")
            errors.append(f"{model_name}: {message[:180]}")
            print(f"Model test failed for {model_name}: {message[:180]}")

    raise RuntimeError(
        "No discovered Gemini Flash model passed generation. "
        + " | ".join(errors[:4])
    )


# --------------------------------------------------
# 5. Fix the model for the entire experiment
# --------------------------------------------------
ACTIVE_LLM_BACKEND = "gemini"
ACTIVE_LLM_MODEL = _test_and_select_gemini()

# Store the exact model name for reproducibility.
os.environ["GEMINI_WORKING_MODEL"] = ACTIVE_LLM_MODEL

print(
    "Fixed answer-generation backend:",
    f"Gemini ({ACTIVE_LLM_MODEL})"
)


# --------------------------------------------------
# 6. Generate answers using the fixed model only
# --------------------------------------------------
def llm_generate(query, idxs, tag=""):
    """
    Generate an answer using the same Gemini model for
    baseline and reranked retrieval.
    Returns: answer, generation_seconds, backend_label
    """
    if ACTIVE_LLM_BACKEND != "gemini" or not ACTIVE_LLM_MODEL:
        raise RuntimeError(
            "Gemini has not been initialized successfully."
        )

    prompt = PROMPT.format(
        ctx=fmt_ctx(idxs),
        q=query,
    )

    start = time.perf_counter()

    response = _gemini_client.models.generate_content(
        model=ACTIVE_LLM_MODEL,
        contents=prompt,
        config={
            "temperature": 0.0,
            "max_output_tokens": 256,
        },
    )

    answer = (response.text or "").strip()
    elapsed = time.perf_counter() - start

    if not answer:
        raise RuntimeError(
            f"Gemini returned an empty answer for {tag or query!r}."
        )

    backend_label = f"Gemini:{ACTIVE_LLM_MODEL}"

    return answer, elapsed, backend_label


print("Cell 7 ready: both pipelines will use the same Gemini model.")


Model test failed for gemini-flash-latest: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to
Model test failed for gemini-2.5-flash: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash fo
Model test failed for gemini-2.5-flash-lite: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash-lite is no longer available to new users. Please update your code to use models/gemini-3.5-fla
Gemini model test passed.
Selected model: gemini-flash-lite-latest
Test response: GEMINI_TEST_OK
Fixed answer-generation backend: Gemini (gemini-flash-lite-latest)
Cell 7 ready: both pipelines will use the same Gemini model.


In [24]:
# Cell 8 — the two pipelines + per-stage latency timers. Run for one query, record everything.
results = []  # one dict per query: ranks, answers, timings (used by the latency table)

def run_query(query, label):
    t = time.perf_counter(); base = baseline_top10(query); t_ret = time.perf_counter() - t
    t = time.perf_counter(); rer = rerank_top10(query, base); t_rer = time.perf_counter() - t
    base3 = [i for i, _ in base[:TOP_K_FINAL]]
    rer3 = [i for i, _ in rer[:TOP_K_FINAL]]
    ans1, t_llm1, be1 = llm_generate(query, base3, 'baseline')
    ans2, t_llm2, be2 = llm_generate(query, rer3, 'reranked')
    rer_pos = {idx: r + 1 for r, (idx, _) in enumerate(rer)}
    rows = [(r + 1, chunks[i].cid, f'{s:.3f}', rer_pos[i], chunks[i].text[:90] + '...')
            for r, (i, s) in enumerate(base)]
    results.append({'label': label, 'query': query,
                    'baseline_top3': [chunks[i].cid for i in base3],
                    'reranked_top3': [chunks[i].cid for i in rer3],
                    'baseline_top10': [chunks[i].cid for i, _ in base],
                    'reranked_top10': [chunks[i].cid for i, _ in rer],
                    'answer1': ans1, 'answer2': ans2,
                    't_ret': t_ret, 't_rer': t_rer,
                    't_llm1': t_llm1, 't_llm2': t_llm2,
                    'backend1': be1, 'backend2': be2})
    print(f"\n{'='*88}\n{label}: {query}\n{'='*88}")
    print(f'{"base_rank":<10}{"chunk":<8}{"cos":<7}{"new_rank":<9}snippet')
    for r in rows:
        mark = '  <-- PROMOTED to Top-3' if r[0] > 3 and r[3] <= 3 else ('' if r[0] <= 3 and r[3] <= 3 else '')
        if r[0] <= 3 or r[3] <= 3:
            print(f'{r[0]:<10}{r[1]:<8}{r[2]:<7}{r[3]:<9}{r[4]}{mark}')
    print(f'\nBaseline Top-3: {[chunks[i].cid for i in base3]} -> Answer 1 ({be1}):\n{ans1}')
    print(f'\nReranked Top-3: {[chunks[i].cid for i in rer3]} -> Answer 2 ({be2}):\n{ans2}')
    return base, rer

## Evaluation queries

The following queries test exceptions, negation/constraints, similar wording with different mechanics, and a rule about when victory is declared. The labels describe the intended test type, not a guaranteed result. Use the printed Top-10 rank changes and answer text to decide whether each example is a real baseline failure that re-ranking fixes.


In [25]:
# Q1 — Exception / corner case: development-card timing.
run_query('Can I play a development card on the same turn I bought it?', 'Q1-exception')



Q1-exception: Can I play a development card on the same turn I bought it?
base_rank chunk   cos    new_rank snippet
1         C047    0.686  1        Cards Y. When you buy a development card, take the top card of the draw pile into your han...
2         C076    0.510  5        ty: If you play this card you may immediately take any 2 resource cards from the supply st...
3         C048    0.501  3        ce. You may not, however , play a card that you bought during the same turn. Exception: If...
4         C031    0.496  2        er prevents it. b) Playing Development Cards Y At any time during your turn, you may play ...  <-- PROMOTED to Top-3

Baseline Top-3: ['C047', 'C076', 'C048'] -> Answer 1 (Gemini:gemini-flash-lite-latest):
Generally, you may not play a development card during the same turn that you bought it [C047, C048]. However, an exception is made if you buy a victory point card that brings you to 10 points, allowing you to immediately reveal it and win the game [C048].

Re

([(47, 0.6855398416519165),
  (76, 0.5098775625228882),
  (48, 0.5010302066802979),
  (31, 0.495683878660202),
  (16, 0.46549373865127563),
  (77, 0.45478230714797974),
  (38, 0.4491676688194275),
  (57, 0.4490472674369812),
  (28, 0.42806118726730347),
  (30, 0.41425544023513794)],
 [(47, 8.982526779174805),
  (31, 8.632223129272461),
  (48, 5.934624195098877),
  (16, 5.761916637420654),
  (76, 5.680839538574219),
  (28, 4.412801742553711),
  (77, 3.9919705390930176),
  (57, 1.3932832479476929),
  (38, 0.73673015832901),
  (30, 0.6010960340499878)])

In [26]:
# Q2 — Negation / constraint: desert production and building.
run_query('Can I build a settlement on the desert, and does the desert ever produce resources?', 'Q2-negation')



Q2-negation: Can I build a settlement on the desert, and does the desert ever produce resources?
base_rank chunk   cos    new_rank snippet
1         C013    0.513  2        Brick Forest Produces Lumber Mountains Produce Ore Fields Produce Grain Pasture Produces W...
2         C046    0.458  1        7 D deser T The desert is the only terrain hex that does not produce resources. The robber...
3         C132    0.396  3        ep harbor. Since it’s impossible for you to have settlements adjacent to all terrain hexes...

Baseline Top-3: ['C013', 'C046', 'C132'] -> Answer 1 (Gemini:gemini-flash-lite-latest):
The desert does not produce resources [C013, C046]. The provided rules do not state whether you can build a settlement on the desert.

Reranked Top-3: ['C046', 'C013', 'C132'] -> Answer 2 (Gemini:gemini-flash-lite-latest):
The desert does not produce resources [C046, C13) and yields fewer resources to adjacent settlements or cities compared to other terrain types [C046]. The provided 

([(13, 0.5132013559341431),
  (46, 0.45841288566589355),
  (132, 0.3963932693004608),
  (42, 0.39401209354400635),
  (130, 0.3924890458583832),
  (94, 0.38592904806137085),
  (44, 0.3799269199371338),
  (38, 0.37926536798477173),
  (21, 0.3765715956687927),
  (95, 0.3720322251319885)],
 [(46, 6.220015048980713),
  (13, 0.10010278224945068),
  (132, 0.057581622153520584),
  (95, -0.2970185875892639),
  (42, -2.2761290073394775),
  (21, -2.793217182159424),
  (130, -2.8134593963623047),
  (44, -3.8975236415863037),
  (94, -4.612930774688721),
  (38, -6.925655841827393)])

In [27]:
# Q3 — Similar wording / possible keyword distractor: Monopoly progress card.
run_query('How do I steal resources from other players with the Monopoly progress card?', 'Q3-monopoly')



Q3-monopoly: How do I steal resources from other players with the Monopoly progress card?
base_rank chunk   cos    new_rank snippet
1         C076    0.561  1        ty: If you play this card you may immediately take any 2 resource cards from the supply st...
2         C035    0.559  7        015 Catan GmbH = © 2015 Catan GmbH © 2015 Catan GmbH = © 2015 Catan GmbH © 2015 Catan GmbH...
3         C029    0.534  2        every player who has more than 7 resource cards must select half (rounded down) of his res...
6         C089    0.462  3        down). (2) Then you (the player who rolled the “7”) must move the robber Y to the number t...  <-- PROMOTED to Top-3

Baseline Top-3: ['C076', 'C035', 'C029'] -> Answer 1 (Gemini:gemini-flash-lite-latest):
When you play the Monopoly card, you must name or announce one type of resource [C076, C035]. All other players must then give you all of the resource cards of that specific type that they have in their hands [C076, C035]. If an opponent does 

([(76, 0.5608568787574768),
  (35, 0.5593984723091125),
  (29, 0.5344079732894897),
  (30, 0.5007632970809937),
  (60, 0.46910983324050903),
  (89, 0.4618551731109619),
  (28, 0.4519559442996979),
  (88, 0.45126211643218994),
  (121, 0.45125335454940796),
  (82, 0.4448848068714142)],
 [(76, 4.690443515777588),
  (29, 4.5543646812438965),
  (89, 4.058551788330078),
  (30, 3.7398362159729004),
  (60, 2.8597946166992188),
  (88, 1.947041630744934),
  (35, 1.4812397956848145),
  (82, 0.41897961497306824),
  (28, -1.2097982168197632),
  (121, -1.7014113664627075)])

In [28]:
# Q4 — Rank-shift test: victory timing on another player's turn.
run_query("Do I win immediately if I reach 10 victory points during another player's turn?", 'Q4-victory-timing')



Q4-victory-timing: Do I win immediately if I reach 10 victory points during another player's turn?
base_rank chunk   cos    new_rank snippet
1         C034    0.675  3        of the game if someone else wins. Y ou may play any number of victory point cards during y...
2         C111    0.556  4        is worth 1 victory point. You only reveal your victory point cards when you or someone els...
3         C048    0.554  2        ce. You may not, however , play a card that you bought during the same turn. Exception: If...
4         C052    0.546  1        trades a lumber and an ore for a brick. Note Beth may not trade with Cooper, since it is P...  <-- PROMOTED to Top-3

Baseline Top-3: ['C034', 'C111', 'C048'] -> Answer 1 (Gemini:gemini-flash-lite-latest):
If you reach 10 points when it is not your turn, the game does not end immediately; it continues until any player has 10 points on their turn [C034]. However, you can win immediately during your own turn if you reach 10 points [C034].

([(34, 0.6748917102813721),
  (111, 0.5560442209243774),
  (48, 0.5538045167922974),
  (52, 0.5457221269607544),
  (112, 0.5045827627182007),
  (54, 0.4683244228363037),
  (117, 0.456550270318985),
  (33, 0.44172996282577515),
  (55, 0.4305967390537262),
  (53, 0.4278547167778015)],
 [(52, 8.902641296386719),
  (48, 7.811996936798096),
  (34, 7.175169944763184),
  (111, 5.81470251083374),
  (112, 5.776562690734863),
  (53, 3.095825672149658),
  (33, 2.9613287448883057),
  (54, 1.9839680194854736),
  (55, 0.3606615960597992),
  (117, -1.7286391258239746)])

In [29]:
# Q5 — Development-card limit: two cards in one turn.
run_query('Can I play two development cards in one turn, such as a Knight and a Monopoly card?', 'Q5-card-limit')


Q5-card-limit: Can I play two development cards in one turn, such as a Knight and a Monopoly card?
base_rank chunk   cos    new_rank snippet
1         C047    0.632  4        Cards Y. When you buy a development card, take the top card of the draw pile into your han...
2         C031    0.552  1        er prevents it. b) Playing Development Cards Y At any time during your turn, you may play ...
3         C076    0.542  2        ty: If you play this card you may immediately take any 2 resource cards from the supply st...
4         C012    0.500  3        ayer who plays more Knight cards takes this card. • A City replaces an already-built Settl...  <-- PROMOTED to Top-3

Baseline Top-3: ['C047', 'C031', 'C076'] -> Answer 1 (Gemini:gemini-flash-lite-latest):
No, you may only play 1 development card during your turn [C047, C076]. Therefore, you cannot play both a Knight and a Monopoly card in the same turn [C047, C076].

Reranked Top-3: ['C031', 'C076', 'C012'] -> Answer 2 (Gemini:gemini-f

([(47, 0.6320147514343262),
  (31, 0.5518360733985901),
  (76, 0.5419423580169678),
  (12, 0.5004919767379761),
  (48, 0.4934706687927246),
  (63, 0.4687112271785736),
  (16, 0.4443970322608948),
  (62, 0.43596649169921875),
  (75, 0.43040135502815247),
  (119, 0.4238184988498688)],
 [(31, 6.052831649780273),
  (76, 5.463994026184082),
  (12, 5.429876327514648),
  (47, 5.373494625091553),
  (16, 2.847200393676758),
  (62, -1.883312463760376),
  (48, -1.92241370677948),
  (75, -2.388657331466675),
  (63, -2.8741397857666016),
  (119, -3.1201939582824707)])

In [30]:
# Q6 — Robber rule: choose a victim and handle an empty hand.
run_query('If I move the robber to a hex next to two players, can I choose whom to steal from, and what if that player has no resource cards?', 'Q6-robber-steal')


Q6-robber-steal: If I move the robber to a hex next to two players, can I choose whom to steal from, and what if that player has no resource cards?
base_rank chunk   cos    new_rank snippet
1         C030    0.711  3        e target terrain hex. The player who is robbed holds his resource card hand face down. You...
2         C029    0.708  4        every player who has more than 7 resource cards must select half (rounded down) of his res...
3         C089    0.680  1        down). (2) Then you (the player who rolled the “7”) must move the robber Y to the number t...
4         C060    0.667  2        knight card face up in front of you. You must move the robber away from his current spot a...  <-- PROMOTED to Top-3

Baseline Top-3: ['C030', 'C029', 'C089'] -> Answer 1 (Gemini:gemini-flash-lite-latest):
If the target hex is adjacent to two or more players with settlements or cities, you may choose which one to steal from [C030, C089]. The provided rules do not state what happens if tha

([(30, 0.710597038269043),
  (29, 0.7079099416732788),
  (89, 0.6802570819854736),
  (60, 0.6665481328964233),
  (35, 0.6612056493759155),
  (87, 0.6174726486206055),
  (86, 0.6048436164855957),
  (88, 0.569352388381958),
  (62, 0.5228081941604614),
  (28, 0.521262526512146)],
 [(89, 6.505603313446045),
  (60, 6.2407050132751465),
  (30, 5.742764949798584),
  (29, 5.163807392120361),
  (86, 3.857630491256714),
  (87, 3.161022901535034),
  (62, 3.0746169090270996),
  (88, 1.7612180709838867),
  (35, 1.381113886833191),
  (28, -2.4719810485839844)])

## How to interpret the results

Compare the Top-3 chunk IDs and read the actual chunk snippets before judging quality. A chunk promoted from rank 4–10 into the Top 3 is useful evidence only if it contains information needed to answer the query. Compare Answer 1 and Answer 2 for correctness, completeness, and citation support. The robber query is a promising test: the saved run showed the baseline answer missed the empty-hand condition while re-ranking placed C060 in the Top 3 and the answer included “you get nothing.” Re-run the notebook to verify this result in a fresh execution. Other queries should only be described as failure cases if their fresh outputs show a real difference.


In [31]:
# Final cell — ranking audit + latency profiling.
# These are measured results from this notebook run; rank promotion alone does not prove answer improvement.
print("RANKING AUDIT (Top-3 chunks)")
for r in results:
    promoted = [cid for cid in r.get('reranked_top3', []) if cid not in r.get('baseline_top3', [])]
    dropped = [cid for cid in r.get('baseline_top3', []) if cid not in r.get('reranked_top3', [])]
    print(f"{r['label']}: baseline={r.get('baseline_top3', [])} | reranked={r.get('reranked_top3', [])} | promoted into Top-3={promoted or 'none'} | dropped={dropped or 'none'}")

print("\nLATENCY PROFILE (seconds; answer generation reported separately)")
print(f'{"query":<22}{"baseline ret.":<16}{"cross-enc.":<14}{"advanced total":<17}{"overhead":<11}{"Answer backend"}')
for r in results:
    advanced_total = r['t_ret'] + r['t_rer']
    overhead = advanced_total / max(r['t_ret'], 1e-9)
    backends = f"{r['backend1']} / {r['backend2']}"
    print(f"{r['label']:<22}{r['t_ret']:<16.3f}{r['t_rer']:<14.3f}{advanced_total:<17.3f}{overhead:<10.2f}x {backends}")
if results:
    mean_ret = float(np.mean([r['t_ret'] for r in results]))
    mean_rerank = float(np.mean([r['t_rer'] for r in results]))
    mean_total = mean_ret + mean_rerank
    print(f'\nMean baseline retrieval: {mean_ret:.3f}s')
    print(f'Mean Cross-Encoder re-ranking (10 pairs): {mean_rerank:.3f}s')
    print(f'Mean advanced retrieval stage: {mean_total:.3f}s')
    print(f'Mean added retrieval-stage overhead: {mean_rerank:.3f}s; advanced/baseline ratio: {mean_total / max(mean_ret, 1e-9):.2f}x')
    print('Mean generation times (separate):', f"Answer 1 {np.mean([r['t_llm1'] for r in results]):.3f}s; Answer 2 {np.mean([r['t_llm2'] for r in results]):.3f}s")
print('Corpus size:', len(chunks), 'chunks; Cross-Encoder scores only the Bi-Encoder Top-10 candidates, not the full corpus.')
print('Timing note: single-run timings are illustrative and vary with warm-up, hardware, and API load.')
print('Interpretation rule: report an improvement only when the retrieved evidence and answer support it; a rank change by itself is not enough.')


RANKING AUDIT (Top-3 chunks)
Q1-exception: baseline=['C047', 'C076', 'C048'] | reranked=['C047', 'C031', 'C048'] | promoted into Top-3=['C031'] | dropped=['C076']
Q2-negation: baseline=['C013', 'C046', 'C132'] | reranked=['C046', 'C013', 'C132'] | promoted into Top-3=none | dropped=none
Q3-monopoly: baseline=['C076', 'C035', 'C029'] | reranked=['C076', 'C029', 'C089'] | promoted into Top-3=['C089'] | dropped=['C035']
Q4-victory-timing: baseline=['C034', 'C111', 'C048'] | reranked=['C052', 'C048', 'C034'] | promoted into Top-3=['C052'] | dropped=['C111']
Q5-card-limit: baseline=['C047', 'C031', 'C076'] | reranked=['C031', 'C076', 'C012'] | promoted into Top-3=['C012'] | dropped=['C047']
Q6-robber-steal: baseline=['C030', 'C029', 'C089'] | reranked=['C089', 'C060', 'C030'] | promoted into Top-3=['C060'] | dropped=['C029']

LATENCY PROFILE (seconds; answer generation reported separately)
query                 baseline ret.   cross-enc.    advanced total   overhead   Answer backend
Q1-exce